# Beginner E4: concatenate Whisper embeddings and E1 acoustic features

## What this experiment is trying to do

The project already has two ways to describe a speech clip: **E1 acoustic measurements** such as MFCCs and pitch, and a **Whisper encoder embedding** learned from speech. This notebook asks: **Does giving one classifier both representations improve Singapore-accent classification beyond using either representation alone?**

It extracts both feature types from the **same clips**, joins their processed vectors, and trains a regularized classifier on the combined input. Separate E1-only and Whisper-only classifiers provide a fair comparison. All three use the same speaker-disjoint train, validation, and test sets; validation chooses settings, and the held-out test set measures the result.

A better score would show that concatenation helped on this dataset. It would **not by itself show** that the model learned accent rather than differences between the MNSC and Common Voice recordings.

This notebook is **self-contained**. It reads `Singlish/data/audio` directly, extracts both representations from each clip, makes one speaker-disjoint train/validation/test split, and compares three classifiers:

| Model | Input |
|---|---|
| E1 acoustic | 39 Librosa MFCC, pitch, spectral, zero-crossing, and energy statistics |
| Whisper | Frozen `openai/whisper-base.en` encoder embedding |
| Feature fusion | The two processed feature blocks concatenated into one vector |

This is **feature-level fusion**: one classifier learns from both blocks. Whisper stays frozen. All preprocessing, PCA and classifier fitting respect the speaker split. The cache created here belongs only to this notebook; no other notebook output is needed.

> MNSC versus Common Voice can differ in microphone, prompt and corpus style. A high score alone does not establish accent recognition.


## Before you start

In Colab, select **Runtime → Change runtime type → T4 GPU** if available. Downloading Whisper and extracting embeddings is much faster on a GPU. A CPU works for a small smoke test. Put audio in the same speaker folders used by E1:

```text
Singlish/data/audio/
├── singapore/<speaker>/<clip>.wav
└── non_singapore/<speaker>/<clip>.wav
```

The first run downloads `openai/whisper-base.en`. Start with 100 clips per class, then set `MAX_CLIPS_PER_CLASS = None` for the full dataset.


In [ ]:
%pip -q install "transformers>=4.46,<5" "librosa>=0.10.2" scikit-learn matplotlib joblib soundfile


## 1. Set paths and experiment size

Only `data/audio` must exist before the run. This notebook writes its own cache and results under `outputs/e4_feature_fusion`.


In [ ]:
from pathlib import Path
USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/Singlish')
DATA_ROOT = PROJECT_ROOT / 'data' / 'audio'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'e4_feature_fusion'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 42
MAX_CLIPS_PER_CLASS = 100  # None uses all available clips.
SAMPLE_RATE = 16_000
MAX_SECONDS = 6
MODEL_NAME = 'openai/whisper-base.en'
BATCH_SIZE = 4
PCA_DIMENSIONS = 32
C_VALUES = [0.01, 0.1, 1.0, 10.0]
print('Audio:', DATA_ROOT, '| outputs:', OUTPUT_DIR)


In [ ]:
import json
import random
from collections import Counter
import joblib
import librosa
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from transformers import WhisperFeatureExtractor, WhisperModel
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Whisper device:', DEVICE)


## 2. Select clips across speakers

Label `1` is the Singapore folder; label `0` is the non-Singapore folder. The speaker folder becomes a group ID. Sampling round-robin across speakers gives the small learning set more distinct voices.


In [ ]:
AUDIO_EXTENSIONS = {'.wav', '.mp3', '.flac', '.ogg', '.m4a'}
CLASS_LABELS = {'non_singapore': 0, 'singapore': 1}
def find_recordings(root):
    rows = []
    for class_name, label in CLASS_LABELS.items():
        class_dir = root / class_name
        if not class_dir.is_dir():
            raise FileNotFoundError(f'Missing {class_dir}')
        for path in sorted(class_dir.rglob('*')):
            if not path.is_file() or path.suffix.lower() not in AUDIO_EXTENSIONS:
                continue
            parts = path.relative_to(class_dir).parts
            if len(parts) < 2:
                raise ValueError(f'Put each clip in a speaker folder: {path}')
            rows.append((path, label, f'{class_name}::{parts[0]}'))
    return rows

def select_recordings(rows, limit, seed):
    if limit is None:
        return rows
    if not isinstance(limit, int) or isinstance(limit, bool) or limit < 1:
        raise ValueError('MAX_CLIPS_PER_CLASS must be a positive integer or None')
    rng = random.Random(seed)
    selected = []
    for label in CLASS_LABELS.values():
        by_speaker = {}
        for row in rows:
            if row[1] == label:
                by_speaker.setdefault(row[2], []).append(row)
        speakers = sorted(by_speaker)
        rng.shuffle(speakers)
        for speaker in speakers:
            rng.shuffle(by_speaker[speaker])
        count = 0
        while count < limit:
            added = False
            for speaker in speakers:
                if by_speaker[speaker]:
                    selected.append(by_speaker[speaker].pop())
                    count += 1
                    added = True
                    if count == limit:
                        break
            if not added:
                break
    return sorted(selected, key=lambda row: str(row[0]))

recordings = select_recordings(find_recordings(DATA_ROOT), MAX_CLIPS_PER_CLASS, RANDOM_SEED)
if not recordings:
    raise RuntimeError(f'No audio found under {DATA_ROOT}')
print('Selected clips:', Counter(row[1] for row in recordings))
print('Speakers:', {label: len({row[2] for row in recordings if row[1] == label}) for label in (0, 1)})


## 3. Prepare each waveform **once**

Both branches receive the same mono, 16 kHz speech, trimmed at the edges, peak-normalized and capped at six seconds. E1 features use a one-second minimum for stable statistics. Whisper receives that same waveform. A failed clip is removed from both branches, keeping their rows aligned.


In [ ]:
N_MFCC = 13
N_FFT = 1024
HOP_LENGTH = 256

def load_and_prepare_audio(path):
    y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    y, _ = librosa.effects.trim(y, top_db=30)
    if y.size == 0:
        raise ValueError('No audible signal')
    peak = np.max(np.abs(y))
    if peak > 0:
        y = y / peak
    y = librosa.util.fix_length(y, size=max(len(y), SAMPLE_RATE))
    return np.ascontiguousarray(y[:SAMPLE_RATE * MAX_SECONDS], dtype=np.float32)

def mean_and_std(feature):
    return np.r_[np.mean(feature, axis=1), np.std(feature, axis=1)]

def acoustic_features(y):
    mfcc = librosa.feature.mfcc(y=y, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
                                n_fft=N_FFT, hop_length=HOP_LENGTH)
    rms = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP_LENGTH)
    f0 = librosa.yin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'),
                     sr=SAMPLE_RATE, frame_length=2048, hop_length=HOP_LENGTH)
    energy = rms.ravel()[:len(f0)]
    voiced = np.isfinite(f0) & (energy > max(float(np.median(energy)) * 0.1, 1e-4))
    voiced_f0 = f0[voiced]
    pitch = [float(np.mean(voiced_f0)) if len(voiced_f0) else 0.0,
             float(np.std(voiced_f0)) if len(voiced_f0) else 0.0,
             float(np.mean(voiced))]
    time_features = [
        librosa.feature.spectral_centroid(y=y, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH),
        librosa.feature.spectral_bandwidth(y=y, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH),
        librosa.feature.spectral_rolloff(y=y, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH),
        librosa.feature.zero_crossing_rate(y, frame_length=N_FFT, hop_length=HOP_LENGTH), rms,
    ]
    vector = np.r_[mean_and_std(mfcc), pitch, *[mean_and_std(v) for v in time_features]]
    vector = np.nan_to_num(vector, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    assert vector.shape == (39,)
    return vector


## 4. Extract frozen Whisper embeddings

Whisper's feature extractor pads clips to its 30-second input. The encoder outputs one hidden vector per time step. We average **only the time steps corresponding to actual audio** (about one encoder step per 20 ms), so padded silence does not dominate a 3–6 second clip. The encoder weights never change.

Both feature matrices are cached together. The cache is reused only if selected paths, file sizes, modification times, model and preprocessing settings match. Remove or rebuild it after changing the feature code itself.


In [ ]:
def load_frozen_encoder():
    feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
    model = WhisperModel.from_pretrained(MODEL_NAME).to(DEVICE)
    encoder = model.get_encoder().eval()
    for parameter in encoder.parameters():
        parameter.requires_grad = False
    return feature_extractor, encoder

@torch.inference_mode()
def encode_waveforms(waveforms, feature_extractor, encoder):
    features = feature_extractor(waveforms, sampling_rate=SAMPLE_RATE,
                                 return_tensors='pt').input_features.to(DEVICE)
    hidden = encoder(input_features=features).last_hidden_state
    vectors = []
    for i, waveform in enumerate(waveforms):
        real_steps = min(hidden.shape[1], max(1, (len(waveform) + 319) // 320))
        vectors.append(hidden[i, :real_steps].mean(dim=0).float().cpu().numpy())
    return np.stack(vectors).astype(np.float32)


In [ ]:
CACHE_PATH = OUTPUT_DIR / 'whisper_e1_features.npz'
expected_paths = np.array([str(row[0]) for row in recordings])
expected_signatures = np.array([f'{row[0].stat().st_size}:{row[0].stat().st_mtime_ns}' for row in recordings])
settings = f'{MODEL_NAME}|sr={SAMPLE_RATE}|max_seconds={MAX_SECONDS}|trim_db=30|peak=1|e1_v1|real_audio_pool_v1'
use_cache = False
if CACHE_PATH.exists():
    with np.load(CACHE_PATH, allow_pickle=False) as cache:
        use_cache = (str(cache['settings']) == settings
                     and np.array_equal(cache['selected_paths'], expected_paths)
                     and np.array_equal(cache['file_signatures'], expected_signatures))
        if use_cache:
            A = cache['acoustic'].astype(np.float32)
            W = cache['whisper'].astype(np.float32)
            labels = cache['labels'].astype(np.int64)
            groups = cache['groups'].astype(str)
            paths = cache['paths'].astype(str)
            print('Loaded this notebook\'s feature cache')
if not use_cache:
    feature_extractor, encoder = load_frozen_encoder()
    acoustic_rows, whisper_batches, kept_rows = [], [], []
    for start in range(0, len(recordings), BATCH_SIZE):
        waveforms, features, valid_rows = [], [], []
        for row in recordings[start:start + BATCH_SIZE]:
            try:
                waveform = load_and_prepare_audio(row[0])
                feature = acoustic_features(waveform)
            except Exception as exc:
                print('Skipping', row[0].name, ':', exc)
                continue
            waveforms.append(waveform)
            features.append(feature)
            valid_rows.append(row)
        if waveforms:
            whisper_batches.append(encode_waveforms(waveforms, feature_extractor, encoder))
            acoustic_rows.extend(features)
            kept_rows.extend(valid_rows)
        print(f'Processed {min(start+BATCH_SIZE, len(recordings))}/{len(recordings)}')
    if not kept_rows:
        raise RuntimeError('No usable clips')
    A = np.vstack(acoustic_rows).astype(np.float32)
    W = np.vstack(whisper_batches).astype(np.float32)
    labels = np.array([row[1] for row in kept_rows], dtype=np.int64)
    groups = np.array([row[2] for row in kept_rows])
    paths = np.array([str(row[0]) for row in kept_rows])
    if len(kept_rows) == len(recordings):
        np.savez_compressed(CACHE_PATH, acoustic=A, whisper=W, labels=labels,
                            groups=groups, paths=paths, selected_paths=expected_paths,
                            file_signatures=expected_signatures, settings=np.array(settings))
        print('Saved feature cache:', CACHE_PATH)
    else:
        print('Some clips failed, so the feature cache was not saved')
    del encoder
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
assert A.shape == (len(labels), 39) and W.shape[0] == len(labels)
assert len(labels) == len(groups) == len(paths)
assert np.isfinite(A).all() and np.isfinite(W).all()
print('E1 features:', A.shape, '| Whisper embeddings:', W.shape)


## 5. Make one speaker-disjoint train/validation/test split

The first group fold is the final test set. A second group fold from the remaining speakers is validation. Every representation uses these **same indices**. We tune only on validation and inspect test performance once.


In [ ]:
speaker_counts = {label: len(set(groups[labels == label])) for label in (0, 1)}
if min(speaker_counts.values()) < 5:
    raise ValueError('Use at least five speakers per class for train/validation/test splitting')
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
trainval_idx, test_idx = next(outer.split(A, labels, groups))
inner_counts = {label: len(set(groups[trainval_idx][labels[trainval_idx] == label])) for label in (0, 1)}
inner_folds = min(5, min(inner_counts.values()))
if inner_folds < 2:
    raise ValueError('Too few training speakers for validation')
inner = StratifiedGroupKFold(n_splits=inner_folds, shuffle=True, random_state=RANDOM_SEED + 1)
train_relative, val_relative = next(inner.split(A[trainval_idx], labels[trainval_idx], groups[trainval_idx]))
train_idx, val_idx = trainval_idx[train_relative], trainval_idx[val_relative]
for name, indices in [('train', train_idx), ('validation', val_idx), ('test', test_idx)]:
    assert set(labels[indices]) == {0, 1}, f'{name} must have both labels'
    print(name, 'clips:', len(indices), '| speakers:', len(set(groups[indices])),
          '| labels:', Counter(labels[indices]))
for left, right in [(train_idx, val_idx), (train_idx, test_idx), (val_idx, test_idx)]:
    assert not (set(groups[left]) & set(groups[right]))
print('Speaker overlap across all three sets: 0')


## 6. Scale each block separately; reduce the larger one

E1 has 39 columns while Whisper has hundreds. We fit separate scalers, reduce Whisper with PCA, then standardize the PCA columns. For the fused vector, each block is divided by the square root of its column count so neither block has greater total scale merely because it has more columns. **These transformations fit only on the supplied fitting speakers.**


In [ ]:
def make_feature_views(fit_idx):
    acoustic_scaler = StandardScaler().fit(A[fit_idx])
    a = acoustic_scaler.transform(A)
    whisper_scaler = StandardScaler().fit(W[fit_idx])
    w_scaled = whisper_scaler.transform(W)
    dimensions = min(PCA_DIMENSIONS, len(fit_idx) - 1, W.shape[1])
    if dimensions < 2:
        raise ValueError('Too few fitting clips for PCA')
    whisper_pca = PCA(n_components=dimensions, random_state=RANDOM_SEED).fit(w_scaled[fit_idx])
    w_pca = whisper_pca.transform(w_scaled)
    pca_scaler = StandardScaler().fit(w_pca[fit_idx])
    w = pca_scaler.transform(w_pca)
    fused = np.column_stack((a / np.sqrt(a.shape[1]), w / np.sqrt(w.shape[1])))
    views = {'E1 acoustic': a, 'Whisper': w, 'Concatenated': fused}
    preprocessors = {'acoustic_scaler': acoustic_scaler, 'whisper_scaler': whisper_scaler,
                     'whisper_pca': whisper_pca, 'pca_scaler': pca_scaler}
    return views, preprocessors

validation_views, _ = make_feature_views(train_idx)
for name, matrix in validation_views.items():
    print(name, matrix.shape)


## 7. Choose each classifier's regularization on validation speakers

Each model uses class-balanced logistic regression and the **same C choices**. `C` controls regularization: smaller values constrain the classifier more. We select macro F1 on validation; balanced accuracy breaks ties. The test set is still untouched.


In [ ]:
chosen_c = {}
validation_results = {}
for name, matrix in validation_views.items():
    trials = []
    for c_value in C_VALUES:
        model = LogisticRegression(C=c_value, class_weight='balanced', max_iter=3000,
                                   random_state=RANDOM_SEED)
        model.fit(matrix[train_idx], labels[train_idx])
        predicted = model.predict(matrix[val_idx])
        trials.append({'C': c_value,
                       'macro_f1': float(f1_score(labels[val_idx], predicted, average='macro')),
                       'balanced_accuracy': float(balanced_accuracy_score(labels[val_idx], predicted))})
    best = max(trials, key=lambda row: (row['macro_f1'], row['balanced_accuracy'], -row['C']))
    chosen_c[name] = best['C']
    validation_results[name] = {'selected': best, 'trials': trials}
    print(name, '| C:', best['C'], '| validation macro F1:', round(best['macro_f1'], 3),
          '| balanced accuracy:', round(best['balanced_accuracy'], 3))


## 8. Refit on train + validation, then evaluate test speakers

Now that each `C` is fixed, we refit scaling and PCA on **train + validation**, train each classifier on those speakers, and evaluate the held-out test speakers once. Test metrics are comparable because rows and speakers are identical across all three models.


In [ ]:
final_views, preprocessors = make_feature_views(trainval_idx)
models, test_results = {}, {}
for name, matrix in final_views.items():
    model = LogisticRegression(C=chosen_c[name], class_weight='balanced', max_iter=3000,
                               random_state=RANDOM_SEED)
    model.fit(matrix[trainval_idx], labels[trainval_idx])
    predicted = model.predict(matrix[test_idx])
    positive_prob = model.predict_proba(matrix[test_idx])[:, list(model.classes_).index(1)]
    models[name] = model
    test_results[name] = {
        'macro_f1': float(f1_score(labels[test_idx], predicted, average='macro')),
        'balanced_accuracy': float(balanced_accuracy_score(labels[test_idx], predicted)),
        'roc_auc': float(roc_auc_score(labels[test_idx], positive_prob)),
        'confusion_matrix': confusion_matrix(labels[test_idx], predicted, labels=[0, 1]).tolist(),
    }
    print(name, json.dumps(test_results[name]))


### Compare the three models

The bars show held-out performance. The fusion result should be compared with the **best single branch**, not only the weaker one. With a small test set, a difference of a few clips can move these metrics substantially.


In [ ]:
names = list(test_results)
positions = np.arange(len(names))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(positions - 0.18, [test_results[n]['macro_f1'] for n in names], 0.36, label='Macro F1')
ax.bar(positions + 0.18, [test_results[n]['balanced_accuracy'] for n in names], 0.36,
       label='Balanced accuracy')
ax.set(xticks=positions, xticklabels=names, ylim=(0, 1.05), ylabel='Held-out test score',
       title='Single representations versus feature concatenation')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'fusion_comparison.png', dpi=150)
plt.show()

matrix = np.array(test_results['Concatenated']['confusion_matrix'])
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(matrix, cmap='Blues')
for (row, col), value in np.ndenumerate(matrix):
    ax.text(col, row, str(value), ha='center', va='center')
ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=['Non-SG', 'Singapore'],
       yticklabels=['Non-SG', 'Singapore'], xlabel='Predicted', ylabel='True',
       title='Concatenated model: held-out speakers')
fig.colorbar(image, ax=ax)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'fusion_confusion_matrix.png', dpi=150)
plt.show()


## 9. Save this standalone experiment

The feature cache, split, fitted preprocessors, classifiers, metrics and charts are all produced by **this notebook**. Load saved `joblib` files only from a source you trust.


In [ ]:
split_path = OUTPUT_DIR / 'speaker_split.npz'
model_path = OUTPUT_DIR / 'fusion_models.joblib'
report_path = OUTPUT_DIR / 'fusion_report.json'
np.savez_compressed(split_path, paths=paths, labels=labels, groups=groups,
                    train_idx=train_idx, val_idx=val_idx, test_idx=test_idx)
joblib.dump({'preprocessors': preprocessors, 'models': models,
             'whisper_model': MODEL_NAME, 'chosen_c': chosen_c}, model_path)
report = {'whisper_model': MODEL_NAME, 'pooling': 'mean over real audio steps only',
          'positive_label': 1, 'pca_dimensions_requested': PCA_DIMENSIONS,
          'pca_dimensions_fitted': int(preprocessors['whisper_pca'].n_components_),
          'split': {name: {'clips': int(len(indices)), 'speakers': int(len(set(groups[indices]))),
                           'label_counts': {str(label): int(np.sum(labels[indices] == label)) for label in (0, 1)}}
                    for name, indices in [('train', train_idx), ('validation', val_idx), ('test', test_idx)]},
          'validation': validation_results, 'test': test_results}
report_path.write_text(json.dumps(report, indent=2) + '\n')
print('Saved:', CACHE_PATH, split_path, model_path, report_path, sep='\n')


## How to interpret the outcome

A gain from concatenation suggests that the two representations add useful information **on this split**. If fusion loses to the best single model, the added features may be redundant, noisy, or overfitted. The microphone and corpus differences between MNSC and Common Voice remain a major alternative explanation for high scores. For an accent claim, repeat the comparison on recordings of both groups captured under comparable conditions.
